# Data Cleaning & Preprocessing Notebook Overview

To prepare the dataset for analysis, several preprocessing steps were performed:

- Binary categorical variables (e.g., Partner, Dependents, PaperlessBilling) were converted to boolean values.
- Numerical variables were verified and converted to appropriate numeric types.
- Categorical variables with multiple categories were retained as categorical features.

No missing values were identified.

Ensuring correct data types is important for both efficient analysis and accurate modeling.

In [1]:
import pandas as pd
import numpy as np

## 1. Load Raw Data

In [2]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv").copy()

## 2. Drop Irrelevant Columns

In [3]:
df = df.drop("customerID", axis=1)

## 3. Fix Numeric Types

#### 3.1 TotalCharges
TotalCharges contains whitespace entries that must be cleaned before conversion.
After stripping whitespace, the column is converted to numeric, and missing values (all tenure = 0) are filled with 0.

This ensures all numeric features are correctly typed and ready for modeling.

In [4]:
df["TotalCharges"] = df["TotalCharges"].str.strip()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(0, inplace=True)

C:\Users\marle\AppData\Local\Temp\ipykernel_27488\1032278738.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(0, inplace=True)


In [5]:
df[df["tenure"]==0].head(11)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,0.0,No
753,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,0.0,No
936,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,0.0,No
1082,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,0.0,No
1340,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,0.0,No
3331,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,0.0,No
3826,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,0.0,No
4380,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,0.0,No
5218,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,0.0,No
6670,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,0.0,No


#### 3.2 ChargeTenureInteraction
Captures the relationship between monthly cost and customer tenure.  
This interaction helps models understand that:

- new, high‑charge customers churn more  
- long‑tenure, high‑charge customers churn less  

In [6]:
df["ChargeTenureInteraction"] = df["MonthlyCharges"] * df["tenure"]

## 4. Convert Binary Variables

Several features contain Yes/No values. These are converted into numeric 1/0 for easier modeling.

- gender is mapped to a binary feature (is_male)

- All other binary columns (e.g., Partner, Dependents, PaperlessBilling) are converted using a general rule

This ensures consistent numeric representation across the dataset.

In [7]:
df["is_male"] = df["gender"].map({"Male": 1, "Female": 0})
df = df.drop("gender", axis=1)

In [8]:
binary_cols = [col for col in df.columns if df[col].nunique(dropna=True) == 2]
df[binary_cols] = df[binary_cols].replace({"Yes": 1, "No": 0})


C:\Users\marle\AppData\Local\Temp\ipykernel_27488\2579786032.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[binary_cols] = df[binary_cols].replace({"Yes": 1, "No": 0})


## 5. Create Service-Related Features

#### 5.1 TotalServiceCount

Many churn patterns relate to how many services a customer subscribes to.
Service columns are cleaned (handling “No internet service” and “No phone service”) and summed to create a total count from 0–8.


In [9]:
service_cols = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df[service_cols] = df[service_cols].replace({
    "Yes": 1,
    "No": 0,
    "No internet service": 0,
    "No phone service": 0
})

df["TotalServiceCount"] = df[service_cols].sum(axis=1)

C:\Users\marle\AppData\Local\Temp\ipykernel_27488\1638981002.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[service_cols] = df[service_cols].replace({


#### 5.2 HasInternet

A binary indicator showing whether the customer has any internet service.



In [10]:
df["HasInternet"] = (df["InternetService"] != "No").astype(int)

#### 5.3 HasSecurity

A combined feature capturing whether the customer has any support/protection service
(OnlineSecurity or TechSupport).

In [11]:
df["HasSecurity"] = ((df["OnlineSecurity"] == 1) | (df["TechSupport"] == 1)).astype(int)

## 6. Create Household Features

A simple proxy for household composition:

- Customer = 1

    - \+ Partner (if present)

    - \+ Dependents (if present)

This feature helps distinguish between single customers, couples, and families. Groups that behave differently in churn patterns.

In [12]:
df["HouseholdSize"] = df["Partner"] + df["Dependents"] + 1

## 7. Simplify Payment Method

PaymentMethod is grouped into broader categories:

- Electronic

- Mailed

- AutoPay

This reduces noise and highlights meaningful behavioral differences.


In [13]:
df["PaymentType"] = df["PaymentMethod"].replace({
    "Electronic check": "Electronic",
    "Mailed check": "Mailed",
    "Bank transfer (automatic)": "AutoPay",
    "Credit card (automatic)": "AutoPay"
})


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   SeniorCitizen            7043 non-null   int64  
 1   Partner                  7043 non-null   int64  
 2   Dependents               7043 non-null   int64  
 3   tenure                   7043 non-null   int64  
 4   PhoneService             7043 non-null   int64  
 5   MultipleLines            7043 non-null   int64  
 6   InternetService          7043 non-null   object 
 7   OnlineSecurity           7043 non-null   int64  
 8   OnlineBackup             7043 non-null   int64  
 9   DeviceProtection         7043 non-null   int64  
 10  TechSupport              7043 non-null   int64  
 11  StreamingTV              7043 non-null   int64  
 12  StreamingMovies          7043 non-null   int64  
 13  Contract                 7043 non-null   object 
 14  PaperlessBilling        

In [15]:
df.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,...,MonthlyCharges,TotalCharges,Churn,ChargeTenureInteraction,is_male,TotalServiceCount,HasInternet,HasSecurity,HouseholdSize,PaymentType
0,0,1,0,1,0,0,DSL,0,1,0,...,29.85,29.85,0,29.85,0,1,1,0,2,Electronic
1,0,0,0,34,1,0,DSL,1,0,1,...,56.95,1889.50,0,1936.30,1,3,1,1,1,Mailed
2,0,0,0,2,1,0,DSL,1,1,0,...,53.85,108.15,1,107.70,1,3,1,1,1,Mailed
3,0,0,0,45,0,0,DSL,1,0,1,...,42.30,1840.75,0,1903.50,1,3,1,1,1,AutoPay
4,0,0,0,2,1,0,Fiber optic,0,0,0,...,70.70,151.65,1,141.40,0,1,1,0,1,Electronic


## 7. Save Processed Dataset

The cleaned and feature‑engineered dataset is now ready for splitting and modeling.

In [16]:
df.to_csv("../data/PreprocessedData.csv", index=False)